# Adaptive AM-FM Decomposition of Speech for Parkinson’s Disease Classification
## Random Forest Classification Pipeline using the NeuroVoz Dataset

### Notebook Overview
This notebook implements a machine learning pipeline to distinguish between individuals with **Parkinson's Disease (PD)** and **Healthy Controls (HC)** based on acoustic features extracted from vowel phonations (/a/, /e/, /i/, /o/, /u/). 

The analysis focuses on high-frequency resolution features, specifically exploring variations in the first five harmonics ($H_1$ through $H_5$) derived from a high-resolution sinusoidal model named **eaQHM**, to identify potential vocal biomarkers associated with the pathology.

### Data Processing & Methodology
1.  **Data Ingestion:** Loading raw acoustic results from MATLAB (`.mat`) and merging with external subject metadata.
2.  **Feature Engineering:** Unpacking nested harmonic amplitude and frequency variation vectors into discrete scalar features.
3.  **Imputation & Cleaning:** Manual handling of missing demographic data and string normalization for phonetic identifiers.
4.  **Validation:** Multi-layered audits to ensure label consistency and speaker-independent integrity.
5.  **Model Training:** A **Repeated Nested Speaker-Independent Cross-Validation** (10x5 stratified) approach using a **Random Forest Classifier** to ensure robust performance estimates.
6.  **Evaluation:** Analyzing results at both the **Sample Level** (individual recordings) and **Subject Level** (aggregated speaker diagnosis), alongside **Permutation Importance** for feature interpretability.

### Data Ingestion and Feature Engineering

In [ ]:
import pandas as pd
from scipy.io import loadmat

data = loadmat('neurovoz_paper.mat')

data_aeiou = data['results'].squeeze()
data_aeiou = pd.DataFrame(data_aeiou)
data_aeiou.columns=['centroid_mean', 'centroid_std', 'spectral_flux_mean', 'spectral_flux_max', 'teo_mean', 'teo_std', 'am_fm_corr', 'ampl_var', 'freq_var', 'f0_var', 'SRER', 
                    'jitter', 'jitter_T', 'shimmer', 'spectral_slope', 'f0_entropy', 'name']
file_names = data_aeiou['name'].apply(lambda x: x[0])

data = data_aeiou

# Expand ampl_var and freq_var into separate harmonics
ampl_expanded = data['ampl_var'].apply(lambda x: x.flatten())
freq_expanded = data['freq_var'].apply(lambda x: x.flatten())

data['centroid_mean'] = data['centroid_mean'].apply(lambda x: x[0][0])
data['centroid_std']  = data['centroid_std'].apply(lambda x: x[0][0])
data['spectral_flux_mean'] = data['spectral_flux_mean'].apply(lambda x: x[0][0])
data['spectral_flux_max']  = data['spectral_flux_max'].apply(lambda x: x[0][0])
data['teo_mean']  = data['teo_mean'].apply(lambda x: x[0][0])
data['teo_std']   = data['teo_std'].apply(lambda x: x[0][0])
data['am_fm_corr'] = data['am_fm_corr'].apply(lambda x: x[0][0])

data['f0_var']   = data['f0_var'].apply(lambda x: x[0][0])
data['SRER']     = data['SRER'].apply(lambda x: x[0][0])
data['jitter']   = data['jitter'].apply(lambda x: x[0][0])
data['jitter_T'] = data['jitter_T'].apply(lambda x: x[0][0])
data['shimmer']  = data['shimmer'].apply(lambda x: x[0][0])
data['spectral_slope'] = data['spectral_slope'].apply(lambda x: x[0][0])
data['f0_entropy'] = data['f0_entropy'].apply(lambda x: x[0][0])

# Split the filename into its 3 components: ['HC', 'U2', '0135']
filename_parts = file_names.str.split('_')

# Assign label based on the first part (Group): PD = 1 (pathological), HC = 0 (control)
data['label'] = filename_parts.str[0].map({'PD': 1, 'HC': 0})

# Extract the 4-digit speaker ID (the third part of the filename)
data['speaker'] = filename_parts.str[2] 

# Load the new metadata CSVs
metadata_hc = pd.read_csv("metadata_hc.csv")
metadata_pd = pd.read_csv("metadata_pd.csv")

# Combine HC and PD metadata, keeping only the 'ID' and 'Sex' columns
gender_data = pd.concat([metadata_hc[['ID', 'Sex']], metadata_pd[['ID', 'Sex']]])

# Drop missing values and duplicates (since each speaker has multiple audio rows)
gender_data = gender_data.dropna(subset=['Sex']).drop_duplicates()

# Rename columns to match what the rest of your code expects
gender_data = gender_data.rename(columns={'ID': 'speaker', 'Sex': 'gender'})

# Convert the integer ID to a 4-digit string to match the extracted '0135', '0004' format
gender_data['speaker'] = gender_data['speaker'].astype(int).astype(str).str.zfill(4)

# Convert gender from float (1.0 / 0.0) to standard integer (1 / 0)
gender_data['gender'] = gender_data['gender'].astype(int)

# Ensure 'speaker' is a string in the main dataframe as well
data['speaker'] = data['speaker'].astype(str)

# Merge the dataframes together based on 'speaker'
data = data.merge(gender_data, how="left", on="speaker")

# Drop unused features
data = data.drop(columns=['jitter_T', 'ampl_var', 'freq_var', 'am_fm_corr', 'f0_entropy', 'f0_var', 'SRER']) 

### Harmonic Feature Expansion

In [ ]:
for i in range(5): 
    data[f'ampl_var_H{i+1}'] = ampl_expanded.apply(lambda v: v[i])

for i in range(5):
    data[f'freq_var_H{i+1}'] = freq_expanded.apply(lambda v: v[i])
    
print(data.head())
print(data.shape)
print(data.info())

### Gender Column Cleanup and Imputation

In [ ]:
# Identify which speaker IDs failed the merge
missing_speakers = data[data['gender'].isna()]['speaker'].unique()

# 0068 -> FEMALE
# 0069 -> FEMALE
# 0084 -> FEMALE
# 0121 -> MALE
# 0059 -> FEMALE

print(f"Found {len(missing_speakers)} speakers missing gender info:")
print(missing_speakers)

# missing speakers and their corresponding gender (Female=0, Male=1)
manual_genders = {
    '0068': 0,
    '0069': 0,
    '0084': 0,
    '0121': 1,
    '0059': 0
}

# We map the 'speaker' column to our dictionary, and use fillna to only replace the NaNs
data['gender'] = data['gender'].fillna(data['speaker'].map(manual_genders))

data['gender'] = data['gender'].astype(int)

print("Remaining missing genders:", data['gender'].isna().sum())
print("Gender column data type:", data['gender'].dtype)

print(data.info())

### Data Integrity and Label Validation

In [ ]:
# Extract the group prefix ('PD' or 'HC') directly from the original filenames
group_prefix = file_names.str.split('_').str[0]
expected_label = group_prefix.map({'PD': 1, 'HC': 0})

# Check for mismatches between expected and actual
incorrect = data.loc[data['label'] != expected_label, ['speaker', 'label']]

if incorrect.empty:
    print("All labels are correct according to the filename prefix.")
else:
    print("Incorrectly labeled speakers found:")
    print(incorrect.drop_duplicates())

# Check for consistency: each speaker should have exactly one label
speaker_labels = data.groupby('speaker')['label'].nunique()
bad_speakers = speaker_labels[speaker_labels > 1]

if bad_speakers.empty:
    print("No speaker has inconsistent labels.")
else:
    print("Speakers with inconsistent labels:")
    print(bad_speakers)
    
# Count total gender distribution
print("Total gender distribution (0=Female, 1=Male):")
print(data[['speaker', 'gender']].drop_duplicates()['gender'].value_counts())

# Count per group (PD=1 vs HC=0)
print("\nGender distribution per group:")
print(data[['speaker', 'gender', 'label']].drop_duplicates().groupby(['label', 'gender']).size())

print("\nClass Distribution")
print(data['label'].value_counts())

### Vowel Extraction and Sample Distribution

In [ ]:
# Clean the 'name' column so it contains pure strings instead of arrays/lists
data['name'] = data['name'].apply(lambda x: str(x[0]) if not isinstance(x, str) else str(x))

# Extract the vowel from the 'name' column
data['vowel'] = data['name'].str.split('_').str[1].str[0].str.upper()

# Calculate the number of samples for each vowel
vowel_counts = data['vowel'].value_counts()

print("Number of samples per vowel:")
print(vowel_counts)

print("\nBreakdown of vowels by group (1=PD, 0=HC):")
print(data.groupby(['label', 'vowel']).size())

### Final Preparation for Machine Learning

In [ ]:
# Shuffle the data (optional)
random_state = 42

data = data.sample(frac=1, random_state=42).reset_index(drop=True)

groups = data['speaker']
print(len(groups.unique()), 'unique groups found.')
    
data = data.drop(columns=['speaker', 'name', 'vowel'])

print('Data shape:', data.shape)
print(data.head())
print(data.tail())

total_nans = data.isna().sum().sum()
print("Total NaNs in dataset:", total_nans)

print("NaNs per column:")
print(data.isna().sum())

### Repeated, 10-by-5 stratified (by gender and label) speaker-independent nested cross-validation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, precision_score, recall_score
from sklearn.inspection import permutation_importance
from typing import Tuple


def aggregate_mean_by_group(y: np.ndarray, p: np.ndarray, g: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    g = np.asarray(g)
    uniq = np.unique(g)
    y_g = np.zeros(len(uniq), dtype=int)
    p_g = np.zeros(len(uniq), dtype=float)

    for i, gg in enumerate(uniq):
        idx = np.where(g == gg)[0]
        p_g[i] = float(np.mean(p[idx])) if len(idx) else float('nan')
        y_g[i] = int(np.mean(y[idx]) >= 0.5) if len(idx) else 0
    return y_g, p_g, uniq


data['stratify_key'] = data['label'].astype(str) + "_" + data['gender'].astype(str)

X = data.drop(columns=['label', 'gender', 'stratify_key'])
y = data['label']
y_stratify = data['stratify_key']

N_REPEATS = 5  
N_OUTTER_SPLITS = 10   
N_INNER_SPLITS = 5
base_random_state = 42

metrics_template = ["accuracy", "f1", "auc", "precision", "recall"]
results_sample = {k: [] for k in metrics_template}
results_speaker = {k: [] for k in metrics_template}
conf_matrices_sample = []
conf_matrices_speaker = []
all_perm_importances = [] 

print(f"Starting Random Forest Repeated Nested CV...")

for repeat in range(N_REPEATS):
    current_seed = base_random_state + repeat
    print(f"\n" + "="*110)
    print(f"REPEAT {repeat + 1}/{N_REPEATS} (Seed: {current_seed})")
    print("="*110)

    outer_cv = StratifiedGroupKFold(n_splits=N_OUTTER_SPLITS, shuffle=True, random_state=current_seed)
    inner_cv = StratifiedGroupKFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=current_seed)

    rf = RandomForestClassifier(random_state=current_seed, class_weight='balanced')
    param_grid = {
        "n_estimators": [500, 1000, 1500],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5, 10],
        "max_features": ['sqrt', 'log2']
    }
    
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y_stratify, groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        y_strat_train = y_stratify.iloc[train_idx]
        g_train, g_test = groups[train_idx], groups[test_idx]

        grid_search = GridSearchCV(
            estimator=rf,
            param_grid=param_grid,
            cv=list(inner_cv.split(X_train, y_strat_train, g_train)),
            scoring="roc_auc",
            n_jobs=-1
        )
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        bp = grid_search.best_params_

        # SAMPLE-LEVEL CALCULATIONS
        y_pred_s = best_model.predict(X_test)
        y_proba_s = best_model.predict_proba(X_test)[:, 1]

        s_acc = accuracy_score(y_test, y_pred_s)
        s_f1 = f1_score(y_test, y_pred_s, zero_division=0)
        s_auc = roc_auc_score(y_test, y_proba_s)
        s_prec = precision_score(y_test, y_pred_s, zero_division=0)
        s_rec = recall_score(y_test, y_pred_s, zero_division=0)
        conf_matrices_sample.append(confusion_matrix(y_test, y_pred_s))

        # SPEAKER-LEVEL CALCULATIONS
        y_t_spk, y_p_spk, _ = aggregate_mean_by_group(y_test, y_proba_s, g_test)
        y_pred_spk = (y_p_spk >= 0.5).astype(int)

        sp_acc = accuracy_score(y_t_spk, y_pred_spk)
        sp_f1 = f1_score(y_t_spk, y_pred_spk, zero_division=0)
        sp_auc = roc_auc_score(y_t_spk, y_p_spk) if len(np.unique(y_t_spk)) > 1 else np.nan
        sp_prec = precision_score(y_t_spk, y_pred_spk, zero_division=0)
        sp_rec = recall_score(y_t_spk, y_pred_spk, zero_division=0)
        conf_matrices_speaker.append(confusion_matrix(y_t_spk, y_pred_spk))

        # Store results
        current_metrics_s = [s_acc, s_f1, s_auc, s_prec, s_rec]
        current_metrics_sp = [sp_acc, sp_f1, sp_auc, sp_prec, sp_rec]
        
        for m, val in zip(metrics_template, current_metrics_s): results_sample[m].append(val)
        for m, val in zip(metrics_template, current_metrics_sp): results_speaker[m].append(val)

        print(f"R{repeat+1} Fold {fold+1:02d} | Best Params: {bp}")
        print(f"   [Sample]  AUC: {s_auc:.3f} | Acc: {s_acc:.3f} | F1: {s_f1:.3f} | Prec: {s_prec:.3f} | Rec: {s_rec:.3f}")
        print(f"   [Speaker] AUC: {sp_auc:.3f} | Acc: {sp_acc:.3f} | F1: {sp_f1:.3f} | Prec: {sp_prec:.3f} | Rec: {sp_rec:.3f}")
        print("-" * 110)

        pi = permutation_importance(best_model, X_test, y_test, scoring='roc_auc', n_repeats=5, n_jobs=-1)
        all_perm_importances.append(pd.DataFrame({"feature": X_test.columns, "importance": pi.importances_mean, "fold": (repeat * 10) + fold + 1}))


print("\n" + "="*80)
print(f"FINAL AGGREGATED RESULTS (N={N_REPEATS * N_OUTTER_SPLITS} Folds)")
print("="*80)
for m in metrics_template:
    print(f"{m.capitalize():<10} | Sample: {np.nanmean(results_sample[m]):.4f} ± {np.nanstd(results_sample[m]):.4f} | Speaker: {np.nanmean(results_speaker[m]):.4f} ± {np.nanstd(results_speaker[m]):.4f}")

### Performance Evaluation and Feature Interpretability

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd


def plot_cm(cm_list, title, ax):
    cms = np.array(cm_list)
    mean_cm = np.mean(cms, axis=0)
    std_cm = np.std(cms, axis=0)
    
    annot = np.empty_like(mean_cm).astype(str)
    for i in range(mean_cm.shape[0]):
        for j in range(mean_cm.shape[1]):
            annot[i, j] = f"{mean_cm[i, j]:.1f}\n±{std_cm[i, j]:.1f}"
            
    sns.heatmap(mean_cm, annot=annot, fmt='', cmap='Blues', cbar=False, 
                xticklabels=['Healthy', 'Pathological'], 
                yticklabels=['Healthy', 'Pathological'], ax=ax)
    
    ax.set_title(title, fontsize=14, pad=10)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if len(conf_matrices_sample) > 0:
    plot_cm(conf_matrices_sample, 'Waveform (Sample) Level\nRandom Forest Performance', axes[0])
else:
    axes[0].text(0.5, 0.5, 'No Data', ha='center')

if len(conf_matrices_speaker) > 0:
    plot_cm(conf_matrices_speaker, 'Subject (Speaker) Level\nRandom Forest Aggregated', axes[1])
else:
    axes[1].text(0.5, 0.5, 'No Data', ha='center')

plt.tight_layout()
plt.show()


if 'all_perm_importances' in locals() and len(all_perm_importances) > 0:
    final_perm_importance = pd.concat(all_perm_importances).groupby('feature')['importance'].agg(['mean', 'std']).sort_values(by='mean', ascending=False)
    
    plot_data = final_perm_importance.head(20).sort_values(by='mean', ascending=True)

    plt.figure(figsize=(10, 8))
    plt.barh(
        plot_data.index,
        plot_data['mean'],
        xerr=plot_data['std'], 
        align='center',
        alpha=0.8,
        color='#4c72b0', 
        ecolor='black',
        capsize=3
    )
    plt.xlabel("Decrease in AUC (Permutation Importance)", fontsize=12)
    plt.ylabel("Features", fontsize=12)
    plt.title("Top 20 Features - Random Forest Permutation Importance", fontsize=14)
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance data not found. Run the Random Forest analysis first.")